# Enterprise RAG — Hands-On, Part 10 of 11: Evaluation

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [ ]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

### Setup — recap of state from earlier parts


In [ ]:
# Build the index if it is not already there (same check as Part 4).
from enterprise_rag.ingest.store import collection_stats
try:
    stats = collection_stats("meridian")
    assert stats["chunks"] > 0
except Exception:
    from enterprise_rag.ingest.pipeline import ingest
    print("building index...")
    print(ingest().render())

---
# Part 10 - Evaluation

Three families of check. The third is not a metric - it is a **release gate**.

| Family | Metrics | Answers |
|---|---|---|
| Retrieval | recall@k, MRR | did the right document reach the context? |
| Generation | groundedness, refusal accuracy | did the answer use it honestly? |
| **Security** | **leak rate - must be exactly 0** | did a forbidden document ever surface? |

A retrieval regression is a bug you fix next sprint. **A leak is an incident.** So it blocks the
release outright rather than lowering a score.

In [ ]:
from enterprise_rag.evaluation.harness import load_cases

cases = load_cases()
print(f"{len(cases)} golden cases\n")
for kind in ["quality", "security", "behaviour"]:
    sel = [c for c in cases if c["kind"] == kind]
    print(f"{kind.upper()} ({len(sel)})")
    for c in sel:
        print(f"  {c['id']:<5}{c['user_id']:<26}{c['question'][:58]}")
    print()

22 golden cases

QUALITY (10)
  Q01  u_lena_t1                 What does error MRD-5031 mean and is my data lost?
  Q02  u_marco_t3                A customer is over their DPM ceiling during a launch. Can 
  Q03  u_marco_t3                What was the root cause of the 14 March EU ingest degradat
  Q04  u_jin_us_t3               Northgate Retail says their query started timing out with 
  Q05  u_sofia_am                What service credit percentage does Vertex Financial get i
  Q06  u_sofia_am                What is the maximum discount an Account Executive can appr
  Q07  u_lena_t1                 Which agent version do customers need so that MRD-5031 bac
  Q08  u_marco_t3                Why should I not scale ingest workers first during a backp
  Q09  u_lena_t1                 Tell me about MRD-4290 and how it differs from MRD-5031.
  Q10  u_marco_t3                Why did Vertex Financial lose data on 14 March and what do

SECURITY (9)
  S01  u_lena_t1                 What are the 

In [ ]:
# The security gate. This is the suite that must never go red.
from enterprise_rag.evaluation.harness import run_eval

report = run_eval("enterprise", kinds=["security"])
print("\n" + report.render())

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetr

  PASS  S01   security  u_lena_t1                 recall=-     refused=True 


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetr

  PASS  S02   security  u_marco_t3                recall=-     refused=True 


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetr

...

  PASS  S07   security  u_lena_t1                 recall=-     refused=True 


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetr

  PASS  S09   security  u_dana_ext                recall=-     refused=True 


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetr

  PASS  S08   security  u_ravi_sec                recall=-     refused=False

strategy       : enterprise
cases          : 9
pass rate      : 100%
recall@k       : 0.00
MRR            : 0.00
groundedness   : 1.00
refusal acc.   : 100%
LEAKS          : 0  (gate: must be 0)
distractions   : 0  (precision signal, not a gate)
refusal advis. : none  (security cases that answered instead of refusing; no leak occurred)
p50 latency    : 8979 ms
total cost     : $0.0059


### Does the fancy retrieval actually help?

The honest way to defend an advanced-retrieval claim is to benchmark it. Run the *same* golden set
through every strategy and compare.

> **Caveat worth saying out loud in an interview:** this corpus is 22 documents. Retrieval is easy at
> that size, so most strategies will score near-perfectly and the differences will be small. The value
> here is that the harness *exists* and gates the release - on a 200,000-document corpus the same
> table is what tells you whether HyDE earns its latency.

In [ ]:
# Uncomment to run - roughly 2 minutes and a few cents per strategy.
# from enterprise_rag.evaluation.harness import compare_strategies
# rows = compare_strategies(["dense", "bm25", "hybrid", "enterprise"], kinds=["quality"])
# for r in rows:
#     print(r)
print("See scripts/evaluate.py --compare for the full benchmark run.")

See scripts/evaluate.py --compare for the full benchmark run.


---

**◀ Previous:** [9. Attacking it](part09-attacking-it.ipynb)

**Next ▶:** [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)
